In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
q = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/raw/landing/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(f"/Volumes/{catalog}/raw/landing/claims_raw/")
    .selectExpr("*", "_metadata.file_name as _source_file", "current_timestamp() as _ingest_ts")
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"/Volumes/{catalog}/raw/landing/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.claims")
)
q.awaitTermination()

In [0]:
%sql
select * from claims.silver.claims_quarantine 